In [28]:
from langchain_ibm import ChatWatsonx
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    MessagesPlaceholder,
)

from langchain_core.output_parsers import (
    StrOutputParser,
    JsonOutputParser,
    PydanticOutputParser,
)

from dotenv import load_dotenv
import os

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_ibm import WatsonxEmbeddings
from langchain_chroma import Chroma

from pydantic import BaseModel, Field
from langchain_core.runnables import RunnablePassthrough

import re

from pprint import pprint

from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers import (
    EnsembleRetriever,
    ContextualCompressionRetriever,
    BM25Retriever,
)
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

load_dotenv()

apikey=  os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watson_ai_url = os.getenv("WATSONX_URL")

watson_llm = ChatWatsonx(
    apikey=apikey,
    project_id=project_id,
    watsonx_url=watson_ai_url,
    model_id="ibm/granite-4-h-small",
    max_tokens=2000,
    params = {
        "temperature": 0
    }
)

watson_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url=f"{watson_ai_url}",
    api_key=f"{apikey}",
    project_id=f"{project_id}",
    params={"temperature": 0},
)

# 문서 로드
pdf_path = "./data/2018 서비스·집단 분쟁조정 사례집.pdf"

loader = PyPDFLoader(pdf_path)
docs = loader.load()
print(len(docs)) # 예상 값 : 200
pprint(docs[0].page_content)
pprint(docs[0].metadata)

# 전처리 - 정규식을 이용한 내용 추출
pattern = r"사\s*례\s*\d+.*?사건번호.*?결정일자.*?\d{4}\.\s?\d{1,2}\.\s?\d{1,2}\."

target_text = "".join(docs[10].page_content)
split_text = re.findall(pattern, target_text, re.DOTALL)

parts = []
if split_text: # 패턴이 존재할때 패턴을 기준으로 문서 분리
    parts = re.split(pattern, target_text)

    # 사례번호
    case_no = re.findall(r"례\s?(\d+)\s?사건번호", split_text[0])
    print(case_no[0] if case_no else "사례번호 미검출")
else:
    print("패턴 매칭 결과 없음")

# 사건 번호, 결정일자 추출
class CaseMetadata(BaseModel):
    case_number:str = Field(description="사건번호 예: 2018일나565")
    decision_date:str = Field(description="결정일자 예: 2018. 8. 7")


metadata_prompt = PromptTemplate.from_template(
    """\
        다음은 분쟁 조정 사례에 대한 텍스트입니다.

        - case_number : 사건번호
        - decision_date : 결정일자

        반드시 JSON 으로만 반환하세요
        {case_text}
    """
)

structed_llm = watson_llm.with_structured_output(CaseMetadata)
chain = metadata_prompt | structed_llm

if split_text:
    case_metadata = chain.invoke({
        "case_text" : split_text[0]
    })

    print(case_metadata)
    print(dict(case_metadata))


# 주 문 ~~~ (탐색용: parts 가 분할됐고 두 번째 조각이 있을 때만)
if len(parts) > 1:
    pprint(parts[1])

    m = re.search(r"주 문\n", parts[1])
    if m:
        title = parts[1][:m.span()[0]].replace("\n", "").strip()      # 주문 앞부분 = 제목
        content = parts[1][m.span()[0]:].strip()                       # 주문 이후 = 내용
        pprint(title)
        pprint(content)

# 사례 번호, 사건 번호, 결정일자, 제목은 meta데이터 추가
# 주의: 로드한 docs 를 그대로 두고, 결과는 별도 리스트(processed_docs)에 담는다.
#       (이전 코드는 docs = [] 로 덮어써서 docs[10:-2] 가 빈 리스트가 되어 루프가 한 번도 돌지 않았음)
processed_docs = []
case_metadata = {}

# 사건이 시작되는 페이지 ~ 마지막에서 -2 페이지 까지 반복
for doc in docs[10:-2]:
    split_text = re.findall(pattern, "".join(doc.page_content), re.DOTALL)
    if split_text:  # 새 사례가 시작되는 페이지
        case_metadata = {}

        # 사례 번호 추출
        case_no = re.findall(r"례\s?(\d+)\s?사건번호", split_text[0])
        case_metadata["case_no"] = case_no[0] if case_no else ""

        # 패턴 기준으로 텍스트 분할
        parts = re.split(pattern, "".join(doc.page_content))

        if len(parts) > 1 and re.search(r"주 문\n", parts[1]):
            span = re.search(r"주 문\n", parts[1]).span()
            # 제목 추출 (주문 앞부분)
            case_metadata["title"] = parts[1][:span[0]].replace("\n", "").strip()
            # 내용 추출 후 기존 내용 업데이트 (주문 이후)
            doc.page_content = parts[1][span[0]:].strip()
        else:
            case_metadata["title"] = ""

        # 사건 번호, 결정 일자 추출
        i = 0
        while i < 10:
            try:
                response = chain.invoke({"case_text": split_text[0]})
                for k, v in dict(response).items():
                    case_metadata[k] = v.replace("\n", "").replace(" ", "")
                break
            except Exception:
                i += 1
                continue

        doc.metadata.update(case_metadata)
        processed_docs.append(doc)
    else:
        doc.metadata.update(case_metadata)
        processed_docs.append(doc)

len(processed_docs)

200
'소비자분쟁조정위원회\n2018\n서비스·집단 \n분쟁조정 사례집'
{'author': 'PC_A2',
 'creationdate': '2019-06-05T11:33:24+09:00',
 'creator': 'PScript5.dll Version 5.2.2',
 'moddate': '2019-06-05T11:58:31+09:00',
 'page': 0,
 'page_label': '1',
 'producer': 'Acrobat Distiller 9.0.0 (Windows)',
 'source': './data/2018 서비스·집단 분쟁조정 사례집.pdf',
 'title': '<32303139303630355FBCD2BAF1C0DABFF820BBE7B7CAC1FD5BBCADBAF1BDBA5D5FB3BBC1F65FC6EDC1FDBABB76657231312E687770>',
 'total_pages': 200}
01
case_number='2018일나565' decision_date='2018. 8. 7.'
{'case_number': '2018일나565', 'decision_date': '2018. 8. 7.'}
('\n'
 '세탁 후 갑피 마모 및 경화된 가죽 \n'
 '운동화에 대한 손해배상 요구\n'
 '주 문\n'
 '1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽 \n'
 '운동화, 색상 : 흰색) 1켤레를 반환한다. \n'
 '2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원\n'
 '을 지급한다.\n'
 '이 유\n'
 '1. 기초사실\n'
 '가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색, \n'
 '이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10. \n'
 '피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피 \n'
 '마모 및 

188

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

split_docs = splitter.split_documents(processed_docs)

print(split_docs)

[Document(metadata={'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2019-06-05T11:33:24+09:00', 'author': 'PC_A2', 'moddate': '2019-06-05T11:58:31+09:00', 'title': '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구', 'source': './data/2018 서비스·집단 분쟁조정 사례집.pdf', 'total_pages': 200, 'page': 10, 'page_label': '11', 'case_no': '01', 'case_number': '2018일나565', 'decision_date': '2018.8.7.'}, page_content='주 문\n1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽 \n운동화, 색상 : 흰색) 1켤레를 반환한다. \n2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원\n을 지급한다.\n이 유\n1. 기초사실\n가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색, \n이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10. \n피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피 \n마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였\n으나, 이후에도 경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신\n청인에게 손해배상(세탁비 환급 포함)을 요구하였으며, 피신청인은 세탁과실이 없\n다는 이유로 이를 거부하였다.\n나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.'), Document(metadata={'producer': 'Acr

In [4]:
# 마침표 뒤에 나오는 줄바꿈 문자를 그대로 두고 나머지 줄바꿈 문자만 제거
pprint(split_docs[18].page_content)

text = split_docs[18].page_content

text = re.sub(r"(?<!\.)\n"," ", text)
pprint(text)

('제1장\n'
 '일\n'
 '반\n'
 '분\n'
 '쟁\n'
 '조\n'
 '정\n'
 ' 사\n'
 '례 (\n'
 '서\n'
 '비\n'
 '스 )\n'
 '제1장 일반분쟁조정 사례(서비스) ● 11\n'
 '항공권 운임을 기준으로 봄이 상당하다. \n'
 '신청인이 이용한 항공사의 해당 구간 편도(인천-워싱턴) 항공권 운임은 1인당 \n'
 '1,200,000원 ~ 3,600,000원으로 조회되는데, 이를 적용하여 총 구입가에서 동 금액 \n'
 '및 당일 취소에 따른 취소수수료를 공제하면 잔여금액이 존재하지 아니하는 바, 피신\n'
 '청인은 신청인에게 환급할 금액이 없다고 봄이 상당하나, 귀국구간 공항이용료는 신청\n'
 '인에게 환급함이 상당하므로 1개 공항 이용료를 인천공항의 이용요금 수준인 17,000\n'
 '원으로 산정하여, 1인당 34,000원[17,000원×2(공항수)]을 환급함이 상당할 것으로 \n'
 '판단된다.\n'
 '이상을 종합할 때 피신청인은 2018. 8. 20.까지 신청인에게 102,000원(34,000원\n'
 '×3인)을 지급하고, 만일 피신청인이 위 지급을 지체하면 2018. 8. 21.부터 다 갚는')
('제1장 일 반 분 쟁 조 정  사 례 ( 서 비 스 ) 제1장 일반분쟁조정 사례(서비스) ● 11 항공권 운임을 기준으로 봄이 '
 '상당하다.  신청인이 이용한 항공사의 해당 구간 편도(인천-워싱턴) 항공권 운임은 1인당  1,200,000원 ~ 3,600,000원으로 '
 '조회되는데, 이를 적용하여 총 구입가에서 동 금액  및 당일 취소에 따른 취소수수료를 공제하면 잔여금액이 존재하지 아니하는 바, 피신 '
 '청인은 신청인에게 환급할 금액이 없다고 봄이 상당하나, 귀국구간 공항이용료는 신청 인에게 환급함이 상당하므로 1개 공항 이용료를 '
 '인천공항의 이용요금 수준인 17,000 원으로 산정하여, 1인당 34,000원[17,000원×2(공항수)]을 환급함이 상당할 것으로  '
 '판단된다.\n'
 '이

In [5]:
from copy import deepcopy

final_docs = []

for doc in split_docs:
    new_doc = deepcopy(doc)

    text = re.sub(r"(?<!\.)\n"," ", new_doc.page_content)

    new_doc.page_content = (
        f"### 이 사건은 '{new_doc.metadata['title']}에 대한 사례 입니다. \n\n"
        f"{text}"
    )

    final_docs.append(new_doc)

In [6]:
final_docs[18].page_content

"### 이 사건은 '출국편 노쇼(No show)로 인하여 취소된 항공권에 대한 손해배상 요구에 대한 사례 입니다. \n\n제1장 일 반 분 쟁 조 정  사 례 ( 서 비 스 ) 제1장 일반분쟁조정 사례(서비스) ● 11 항공권 운임을 기준으로 봄이 상당하다.  신청인이 이용한 항공사의 해당 구간 편도(인천-워싱턴) 항공권 운임은 1인당  1,200,000원 ~ 3,600,000원으로 조회되는데, 이를 적용하여 총 구입가에서 동 금액  및 당일 취소에 따른 취소수수료를 공제하면 잔여금액이 존재하지 아니하는 바, 피신 청인은 신청인에게 환급할 금액이 없다고 봄이 상당하나, 귀국구간 공항이용료는 신청 인에게 환급함이 상당하므로 1개 공항 이용료를 인천공항의 이용요금 수준인 17,000 원으로 산정하여, 1인당 34,000원[17,000원×2(공항수)]을 환급함이 상당할 것으로  판단된다.\n이상을 종합할 때 피신청인은 2018. 8. 20.까지 신청인에게 102,000원(34,000원 ×3인)을 지급하고, 만일 피신청인이 위 지급을 지체하면 2018. 8. 21.부터 다 갚는"

In [ ]:
# 로컬 저장 : Chroma
# 메모리 : FAISS

# 이 셀을 여러 번 실행해도 중복이 쌓이지 않도록,
# 적재 전에 해당 컬렉션만 비운다. (db/chroma_db 의 다른 컬렉션은 그대로 보존)
import chromadb

collection_name = "customer_dispute_cases"

client = chromadb.PersistentClient(path="./db/chroma_db")
try:
    client.delete_collection(collection_name)  # 기존 컬렉션 제거
except Exception:
    pass  # 컬렉션이 없으면 무시

vectorstore = Chroma.from_documents(
    documents=final_docs,
    embedding=watson_embedding,
    persist_directory="./db/chroma_db",
    collection_name=collection_name,
)

# 적재 후 문서 수 확인 (final_docs 길이와 같아야 정상)
print("적재된 문서 수:", vectorstore._collection.count())
print("final_docs 길이:", len(final_docs))

In [8]:
query = "세탁 후 오염에 대한 손해배상은 어떻게 이루어지는가요?"
similarity_docs = vectorstore.similarity_search(query, k=5)

pprint(similarity_docs)


[Document(id='7413c253-f07c-4f65-b1dd-3f681d6c6ea4', metadata={'author': 'PC_A2', 'case_number': '2018일나565', 'total_pages': 200, 'title': '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구', 'source': './data/2018 서비스·집단 분쟁조정 사례집.pdf', 'page': 11, 'creationdate': '2019-06-05T11:33:24+09:00', 'decision_date': '2018.8.7.', 'case_no': '01', 'moddate': '2019-06-05T11:58:31+09:00', 'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'page_label': '12', 'creator': 'PScript5.dll Version 5.2.2'}, page_content="### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례 입니다. \n\n탁과실로 단정할 수 없다고 판단된 점, 신청인이 착화 과정에서 이 사건 제품을 훼손 하여 그 손해가 발생 및 확대되었을 가능성을 배제할 수 없는 점 등에 비추어 볼 때,  손해의 공평·타당한 분담이라는 손해배상 제도의 지도이념과 상호 양보를 통한 분쟁의  원만한 해결이라는 조정의 취지를 고려하여, 피신청인의 책임을 60%로 제한함이 상당 하다.\n한편, 세탁비와 관련하여,「세탁업 표준약관」제9조 제1항 및 제2항에서는 세탁업자의  책임있는 사유로 세탁물이 손상, 색상변화, 얼룩 등의 하자가 발생였을 때에는 해당  세탁물에 대하여 세탁업자는 고객에게 세탁요금을 청구하지 못하므로, 세탁업자인 피 신청인이 세탁비 4,000원을 신청인에게 환급이 상당하다.\n이상을 종합하면, 신청인은 피신청인에게 이 사건 제품을 반환하고 피신청인은 손해배 상액 67,000원(112,140원 × 60%, 1,0

In [9]:
for doc in similarity_docs:
    print(doc.metadata['case_no'], doc.metadata['page'], doc.page_content[:500])
    print("-------")

01 11 ### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례 입니다. 

탁과실로 단정할 수 없다고 판단된 점, 신청인이 착화 과정에서 이 사건 제품을 훼손 하여 그 손해가 발생 및 확대되었을 가능성을 배제할 수 없는 점 등에 비추어 볼 때,  손해의 공평·타당한 분담이라는 손해배상 제도의 지도이념과 상호 양보를 통한 분쟁의  원만한 해결이라는 조정의 취지를 고려하여, 피신청인의 책임을 60%로 제한함이 상당 하다.
한편, 세탁비와 관련하여,「세탁업 표준약관」제9조 제1항 및 제2항에서는 세탁업자의  책임있는 사유로 세탁물이 손상, 색상변화, 얼룩 등의 하자가 발생였을 때에는 해당  세탁물에 대하여 세탁업자는 고객에게 세탁요금을 청구하지 못하므로, 세탁업자인 피 신청인이 세탁비 4,000원을 신청인에게 환급이 상당하다.
이상을 종합하면, 신청인은 피신청인에게 이 사건 제품을 반환하고 피신청인은 손해배 상액 67,000원(112,140원 × 60%,
-------
01 11 ### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례 입니다. 

탁과실로 단정할 수 없다고 판단된 점, 신청인이 착화 과정에서 이 사건 제품을 훼손 하여 그 손해가 발생 및 확대되었을 가능성을 배제할 수 없는 점 등에 비추어 볼 때,  손해의 공평·타당한 분담이라는 손해배상 제도의 지도이념과 상호 양보를 통한 분쟁의  원만한 해결이라는 조정의 취지를 고려하여, 피신청인의 책임을 60%로 제한함이 상당 하다.
한편, 세탁비와 관련하여,「세탁업 표준약관」제9조 제1항 및 제2항에서는 세탁업자의  책임있는 사유로 세탁물이 손상, 색상변화, 얼룩 등의 하자가 발생였을 때에는 해당  세탁물에 대하여 세탁업자는 고객에게 세탁요금을 청구하지 못하므로, 세탁업자인 피 신청인이 세탁비 4,000원을 신청인에게 환급이 상당하다.
이상을 종합하면, 신청인은 피신청인에게 이 사건 제품을 반환하고 피신청인은 손해배 상액 67

In [12]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5, "filter" : {"case_no" : "01"}})

retrieved_docs= retriever.invoke(query)

for doc in retrieved_docs:
    print(doc.metadata['case_no'], doc.metadata['page'], doc.page_content[:500])
    print("-------")

01 11 ### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례 입니다. 

탁과실로 단정할 수 없다고 판단된 점, 신청인이 착화 과정에서 이 사건 제품을 훼손 하여 그 손해가 발생 및 확대되었을 가능성을 배제할 수 없는 점 등에 비추어 볼 때,  손해의 공평·타당한 분담이라는 손해배상 제도의 지도이념과 상호 양보를 통한 분쟁의  원만한 해결이라는 조정의 취지를 고려하여, 피신청인의 책임을 60%로 제한함이 상당 하다.
한편, 세탁비와 관련하여,「세탁업 표준약관」제9조 제1항 및 제2항에서는 세탁업자의  책임있는 사유로 세탁물이 손상, 색상변화, 얼룩 등의 하자가 발생였을 때에는 해당  세탁물에 대하여 세탁업자는 고객에게 세탁요금을 청구하지 못하므로, 세탁업자인 피 신청인이 세탁비 4,000원을 신청인에게 환급이 상당하다.
이상을 종합하면, 신청인은 피신청인에게 이 사건 제품을 반환하고 피신청인은 손해배 상액 67,000원(112,140원 × 60%,
-------
01 11 ### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례 입니다. 

탁과실로 단정할 수 없다고 판단된 점, 신청인이 착화 과정에서 이 사건 제품을 훼손 하여 그 손해가 발생 및 확대되었을 가능성을 배제할 수 없는 점 등에 비추어 볼 때,  손해의 공평·타당한 분담이라는 손해배상 제도의 지도이념과 상호 양보를 통한 분쟁의  원만한 해결이라는 조정의 취지를 고려하여, 피신청인의 책임을 60%로 제한함이 상당 하다.
한편, 세탁비와 관련하여,「세탁업 표준약관」제9조 제1항 및 제2항에서는 세탁업자의  책임있는 사유로 세탁물이 손상, 색상변화, 얼룩 등의 하자가 발생였을 때에는 해당  세탁물에 대하여 세탁업자는 고객에게 세탁요금을 청구하지 못하므로, 세탁업자인 피 신청인이 세탁비 4,000원을 신청인에게 환급이 상당하다.
이상을 종합하면, 신청인은 피신청인에게 이 사건 제품을 반환하고 피신청인은 손해배 상액 67

In [13]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5, "where_document": {"$contains": "세탁"}}
)

retrieved_docs = retriever.invoke(query)

for doc in retrieved_docs:
    print(doc.metadata["case_no"], doc.metadata["page"], doc.page_content[:500])
    print("-------")

01 11 ### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례 입니다. 

탁과실로 단정할 수 없다고 판단된 점, 신청인이 착화 과정에서 이 사건 제품을 훼손 하여 그 손해가 발생 및 확대되었을 가능성을 배제할 수 없는 점 등에 비추어 볼 때,  손해의 공평·타당한 분담이라는 손해배상 제도의 지도이념과 상호 양보를 통한 분쟁의  원만한 해결이라는 조정의 취지를 고려하여, 피신청인의 책임을 60%로 제한함이 상당 하다.
한편, 세탁비와 관련하여,「세탁업 표준약관」제9조 제1항 및 제2항에서는 세탁업자의  책임있는 사유로 세탁물이 손상, 색상변화, 얼룩 등의 하자가 발생였을 때에는 해당  세탁물에 대하여 세탁업자는 고객에게 세탁요금을 청구하지 못하므로, 세탁업자인 피 신청인이 세탁비 4,000원을 신청인에게 환급이 상당하다.
이상을 종합하면, 신청인은 피신청인에게 이 사건 제품을 반환하고 피신청인은 손해배 상액 67,000원(112,140원 × 60%,
-------
01 11 ### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례 입니다. 

탁과실로 단정할 수 없다고 판단된 점, 신청인이 착화 과정에서 이 사건 제품을 훼손 하여 그 손해가 발생 및 확대되었을 가능성을 배제할 수 없는 점 등에 비추어 볼 때,  손해의 공평·타당한 분담이라는 손해배상 제도의 지도이념과 상호 양보를 통한 분쟁의  원만한 해결이라는 조정의 취지를 고려하여, 피신청인의 책임을 60%로 제한함이 상당 하다.
한편, 세탁비와 관련하여,「세탁업 표준약관」제9조 제1항 및 제2항에서는 세탁업자의  책임있는 사유로 세탁물이 손상, 색상변화, 얼룩 등의 하자가 발생였을 때에는 해당  세탁물에 대하여 세탁업자는 고객에게 세탁요금을 청구하지 못하므로, 세탁업자인 피 신청인이 세탁비 4,000원을 신청인에게 환급이 상당하다.
이상을 종합하면, 신청인은 피신청인에게 이 사건 제품을 반환하고 피신청인은 손해배 상액 67

In [17]:
def format_docs(docs):
    """Document 객체에서 page_content 추출"""
    return "\n\n".join([d.page_content for d in docs])


retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5, "where_document": {"$contains": "세탁"}}
)

# 프롬프트 "객체"를 만들 때는 from_messages 를 사용한다.
# (format_messages 는 이미 만들어진 프롬프트를 포맷팅할 때 쓰는 메서드)
rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "다음 컨텍스트를 참고하여 질문에 답하세요\n. 컨텍스트에 없는 내용은 모른다고 답할 것.\n\n 컨텍스트:\n {context}",
        ),
        ("human", "{query}"),
    ]
)

# 질문 => 임베딩화 => 가장 비슷한 chunks 찾기 => Document 객체 => format_docs => context => llm context 기반으로 답변 생성

chain = (
    {"context": retriever | format_docs, "query": RunnablePassthrough()}
    | rag_prompt
    | watson_llm
    | StrOutputParser()
)

query = "세탁 후 오염에 대한 손해 배상은 어떻게 이루어지는가?"
response = chain.invoke(query)
print(response)

세탁 후 오염에 대한 손해 배상은 일반적으로 세탁업자와 고객 간의 계약 조건, 세탁업 표준약관, 그리고 관련 법규에 따라 이루어집니다. 세탁업자는 고객의 옷을 세탁하는 과정에서 발생할 수 있는 손해에 대해 일정한 책임을 지게 됩니다. 다음은 세탁 후 오염에 대한 손해 배상이 어떻게 이루어지는지에 대한 일반적인 원칙입니다:

1. **계약 조건**: 세탁업자와 고객 간의 계약에는 세탁 서비스의 범위, 책임, 그리고 손해 배상에 관한 조항이 포함될 수 있습니다. 이러한 조항은 세탁업자가 어떤 상황에서 손해 배상을 해야 하는지, 그리고 어떤 상황에서는 책임을 지지 않는지를 명확히 합니다.

2. **세탁업 표준약관**: 많은 국가와 지역에서는 세탁업자가 따라야 하는 표준약관을 제정하고 있습니다. 이러한 표준약관은 세탁업자의 책임 범위, 손해 배상 조건, 그리고 고객의 권리를 규정합니다. 예를 들어, 세탁업 표준약관은 세탁업자가 고객의 옷을 세탁하는 과정에서 발생한 오염이나 손상에 대해 어떤 조건 하에 책임을 지는지를 명시할 수 있습니다.

3. **법적 책임**: 세탁업자는 고객의 옷을 세탁하는 과정에서 발생한 손해에 대해 법적 책임을 지게 될 수 있습니다. 이는 세탁업자가 고객의 옷을 세탁하는 과정에서 합리적인 주의를 기울였음에도 불구하고 손해가 발생한 경우에 해당합니다. 법적 책임은 세탁업자가 고객에게 손해 배상을 해야 하는 의무를 의미합니다.

4. **과실 여부**: 세탁 후 오염이 발생한 경우, 세탁업자와 고객 각각의 과실 여부가 손해 배상 책임에 영향을 미칠 수 있습니다. 예를 들어, 고객이 세탁 전에 옷에 묻은 오염을 제거하지 않았다면, 이로 인해 발생한 손해에 대해 고객이 일부 책임을 지게 될 수 있습니다. 반면, 세탁업자가 세탁 과정에서 과실이 있었다면, 세탁업자가 손해 배상을 해야 할 수 있습니다.

5. **손해 배상 금액**: 손해 배상 금액은 일반적으로 손해가 발생한 옷의 가치, 세탁료, 그리고 고객이 입은 불편 등을 고려하여 결정됩니다. 손

In [19]:
bm25_retriever = BM25Retriever.from_documents(final_docs)
bm25_retriever.k = 5

docs = bm25_retriever.invoke("가죽 운동화 세탁 손해배상")

for doc in docs:
    print(doc.metadata['case_no'], doc.page_content[:500])
    print("=======")

01 ### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례 입니다. 

4 ● 2018 서비스·집단 분쟁조정 사례집 2 .  판   단 신청인은 피신청인의 세탁 후 신발의 갑피가 마모되고 경화되는 현상이 발생하였고,  재세탁 이후에도 경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않았으며, 이  사건 제품보다 더 오래신은 비슷한 흰색 가죽 운동화의 상태보다 이 사건 제품의 상태 가 더 좋지 않기 때문에 본인의 착화습관이 이 사건 현상의 원인이라고 볼 수도 없으 므로, 세탁과실에 따른 손해배상 및 세탁비 환급을 요구한다.
이에 대하여 피신청인은 이 사건 제품을 인수하였을 당시 이미 제품 상태가 좋지 않았 고, 한국소비자원의 신발제품심의위원회에서도 세탁과실로 인정하지 않았기 때문에 신 청인의 요구를 수용할 수 없으나, 다만 원만한 해결을 위해 피해구제 담당자가 제시한  배상산정액 112,140원의 50%인 50,670원을 환급할 의사는 있다고
01 ### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례 입니다. 

나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.
    신청인이 주장하는 갑피 벗겨짐(스크래치 등) 증상은 관찰되나 현 제품 상태만 으로는 제품 훼손의 원인이 세탁 과정상 발생한 것인지 착화 환경에 따른 문제 인지 단정하기 어려운바, 판단 불가하다.
01 ### 이 사건은 '세탁 후 갑피 마모 및 경화된 가죽 운동화에 대한 손해배상 요구에 대한 사례 입니다. 

살피건대, 피신청인은 세탁 전부터 이 사건 제품의 상태가 좋지 않았다고 주장하나,  다음과 같은 사정들, 즉 ① 피신청인이 세탁을 위해 이 사건 제품을 인수하면서 세탁 물의 하자유무가 작성된 인수증을 교부하지 않은 점, ② 「세탁업 표준약관」 제3조 제1 항은 인수증 미교부로 인해 발생한 손해 및 그에 따른 손해배상책임은 세탁업자에게  귀속되는 것으로 규정하고 있는 점, ③ 인수 당시 이 사

In [24]:
ensemble_retriever = EnsembleRetriever(retrievers=[bm25_retriever, retriever], weights=[0.7, 0.3])

docs = ensemble_retriever.invoke("가상화폐 거래소 해킹에 따른 원상회복 요구 사례")

for doc in docs:
    print(doc.metadata['case_no'], doc.page_content[:500])
    print("=======")

05 ### 이 사건은 '가상화폐 거래소 해킹에 따른 원상회복 등 요구에 대한 사례 입니다. 

폐 일부를 탈취 당하였고, 이후 피신청인은 해킹된 가상화폐 중 복구할 수 없는 가상 화폐에 대하여 2가지 보상방안[(1안) 피신청인이 △△△△ 거래소를 운영하면서 발생 하는 이익으로 가상화폐를 매입하여 보상하는 방안, (2안) △△△△의 기축통화이자 지
05 ### 이 사건은 '가상화폐 거래소 해킹에 따른 원상회복 등 요구에 대한 사례 입니다. 

제2장 집 단 분 쟁 조 정  사 례 제2장 집단분쟁조정 사례 ● 187 이상을 종합하면, 피신청인은 2019. 4. 15.까지 가상화폐 △△△을 보유하지 않은 별 지 제1목록에 기재된 32명에게 미복구된 가상화폐에 대하여 해킹 당시 마지막 가상화 폐 거래를 기준으로 가치를 환산하여 손해를 배상하고, 가상화폐 △△△을 보유한 별 지 제2목록에 기재된 신청인 13명의 경우 피신청인이 가상화폐 △△△에 대해서는  2019. 4. 15.까지 복구하며, 같은 날까지 위 △△△을 제외한 나머지 미복구 가상화폐 에 대하여 해킹 당시 마지막 가상화폐 거래를 기준으로 가치를 환산하여 배상하고, 만 약 위 기간까지 가상화폐 △△△에 대하여 복구하지 못하는 경우에는 △△△에 대해서 도 해킹 당시 마지막 가상화폐 거래를 기준으로 가치를 환산하여 2019. 4. 16.까지  신청인들에게 배상하며 피신청인과 별지 제
05 ### 이 사건은 '가상화폐 거래소 해킹에 따른 원상회복 등 요구에 대한 사례 입니다. 

주 문 1. 피신청인은 2019. 4. 15까지 별지 제1목록에 기재된 신청인들에게 같은 목록에 기 재된 손해배상금을 각 지급한다.
2. 피신청인은 2019. 4. 15.까지 별지 제2목록에 기재된 신청인들에게 가상화폐 ○○ ○을 각 복구하고, 같은 목록에 기재된 ○○○을 제외한 손해배상금을 각 지급한다.
3. 만일 피신청인이 제2항의 가상화폐 ○○○을 2019. 4. 15.까지 복구하지 못하는  경우, 같은 목록에 기재된 ○○○을 포함

### SelfQuery



In [26]:
metadata_field_info = [
    AttributeInfo(
        name="case_no",
        description="사건 번호",
        type="string",
    ),
    AttributeInfo(
        name="title",
        description="사건 제목",
        type="string",
    ),
    AttributeInfo(
        name="decision_date",
        description="결정일자",
        type="string",
    ),
]

self_retriever = SelfQueryRetriever.from_llm(
    llm=watson_llm,
    vectorstore=vectorstore,
    document_contents="소비자 분쟁 사례",
    metadata_field_info=metadata_field_info,
    structured_query_translator=ChromaTranslator()
)

docs = self_retriever.invoke("32번 사례를 보여줘라")
pprint(docs)

[Document(id='d34db58e-9fa5-4c35-a4f7-a0721973caa5', metadata={'page_label': '97', 'decision_date': '2017.9.26.', 'moddate': '2019-06-05T11:58:31+09:00', 'case_no': '32', 'author': 'PC_A2', 'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'title': '식당에서 분실된 신발에 대한 배상 요구', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2019-06-05T11:33:24+09:00', 'page': 96, 'source': './data/2018 서비스·집단 분쟁조정 사례집.pdf', 'case_number': '2017일나273', 'total_pages': 200}, page_content="### 이 사건은 '식당에서 분실된 신발에 대한 배상 요구에 대한 사례 입니다. \n\n대물에 대해 책임이 없음을 알린 경우에도 물건의 멸실로 인한 손해를 배상할 책임을  면하지 못하므로, 신청인에게 운동화 분실에 따른 손해를 배상하여야 한다.\n다만, 신청인은 자신의 신발을 누구나 접근할 수 있는 개방된 신발장에 비치하면서 분 실 가능성이 있음을 충분히 예상할 수 있었고, 피신청인이 비치한 비닐봉지를 이용하 여 자신의 신발을 다른 신발과 구분하는 등 주의를 기울일 필요가 있었음에도 어떠한  조치도 취하지 아니하였으므로, 이러한 신청인의 부주의를 고려하여 피신청인의 책임 을 50%로 제한하기로 한다.  그렇다면, 피신청인은 2017. 12. 26.까지 신청인에게 「소비자분쟁해결기준」에 따라  산정한 배상액 65,400원의 50%에 해당하는 32,700원을 지급하고, 만일 지급을 지체 하면 위 돈에 대하여 2017. 12. 27.부터 다 갚는 날까지 「상법」제54조에 따라 연 6% 의 비율에 의한 지연손해금을 가산하여 지

In [27]:
compressor = LLMChainExtractor.from_llm(watson_llm)

compression_retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=ensemble_retriever)

docs = compression_retriever.invoke("가죽 운동화 세탁 손해배상")

for doc in docs:
    print(doc.metadata['case_no'], doc.page_content[:500])
    print("=======")

NameError: name 'LLMChainExtractor' is not defined